In [1]:
import numpy as np
import pandas as pd
import random
import csv
import geopandas as gpd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, MaxAbsScaler
from sklearn.impute import KNNImputer
import sys
import pickle

sys.path.insert(0, "../../../Modules")

pd.set_option('display.max_columns', None)
random.seed(0)
np.random.seed(0)

from utils import *

enc = 'utf-8'
shapefiles_folder = "C:/Users/dimit/Documents/noa hoard/Italy Shapefiles"

c:\Users\dimit\AppData\Local\Programs\Python\Python39\lib\site-packages\geopandas\_compat.py:112: UserWarning: The Shapely GEOS version (3.10.3-CAPI-1.16.1) is incompatible with the GEOS version PyGEOS was compiled with (3.10.4-CAPI-1.16.2). Conversions between both will be slow.
  warnings.warn(


In [2]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
NUTS0 = 'IT'
NUTS2 = 'Veneto'

In [4]:
YEAR = 2023
MONTH = 'June'
PERIOD = '2nd'

In [5]:
model = pickle.load(open(f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_WNV_Linear_model.pkl', 'rb'))
scaler = pickle.load(open(f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_WNV_Scaler.pkl', 'rb'))
imputer = pickle.load(open(f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_WNV_Imputer.pkl', 'rb'))

In [6]:
data_test = read_data(f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_WNV_Dataset_{YEAR}-{MONTH}-{PERIOD}.csv')
data_test.head()

,x,y,dt_placement,nuts2,lau1,day,month,week,year,day_sin,day_cos,month_sin,month_cos,week_sin,week_cos,population,eq_distance,ndvi,ndmi,ndwi,ndbi,ndvi_mean,ndmi_mean,ndwi_mean,ndbi_mean,ndvi_std,ndmi_std,ndwi_std,ndbi_std,lst,lst_day,lst_night,lst_jan_day_mean,lst_jan_night_mean,lst_feb_day_mean,lst_feb_night_mean,lst_mar_day_mean,lst_mar_night_mean,lst_apr_day_mean,lst_apr_night_mean,acc_rainfall_1week,acc_rainfall_2week,acc_rainfall_jan,distance_to_coast,distance_to_river,slope_mean_1km,aspect_mean_200m,elevation_mean_1km,hillshade_mean_1km,fs_area_1km,flow_accu_200m,lc_prop1,lc_prop1_assessment,lc_prop2,lc_prop2_assessment,lc_prop3,lc_prop3_assessment,lc_type1,lc_type2,lc_type3,lc_type4,lc_type5,lw,qc,mosq_pred,mosq_previous,case
0,11.79324,45.35865,2023-06-16,veneto,abano terme,16,6,24,2023,-0.101168,-0.994869,1.224647e-16,-1.0,0.292057,-0.956401,"19,349",46.86670,0.422233,0.133445,-0.375960,-0.133445,0.342973,0.086453,-0.322667,-0.086453,0.065059,0.040007,0.049200,0.040007,27.070,34.410,19.73,7.626000,0.600000,8.080000,-0.791538,18.365714,6.326364,21.500769,6.950000,18.414280,71.088260,807.787942,8982.126981,2074.539971,3,170.409233,10.722186,179.966181,0.0,1.063084,31,98.0,36,98.0,30,98.0,12,12,1,6,7,2,0,194,0,0
1,12.04199,45.06525,2023-06-16,veneto,adria,16,6,24,2023,-0.101168,-0.994869,1.224647e-16,-1.0,0.292057,-0.956401,"20,233",46.64640,0.574252,0.233525,-0.494904,-0.233525,0.446317,0.128807,-0.410099,-0.128807,0.134495,0.104434,0.090522,0.104434,24.486,32.222,16.75,6.377211,0.808571,8.476395,-1.590923,16.728553,4.271543,21.836459,5.637644,25.051933,86.014148,1219.561493,6701.380530,1644.125334,2,145.603087,-1.354496,180.070371,0.0,1.206604,31,99.0,36,99.0,30,99.0,12,12,1,6,7,2,0,292,0,0
2,10.77673,45.55680,2023-06-16,veneto,affi,16,6,24,2023,-0.101168,-0.994869,1.224647e-16,-1.0,0.292057,-0.956401,"2,297",46.81410,0.348467,0.056085,-0.341872,-0.056085,0.369168,0.141192,-0.325122,-0.141192,0.087405,0.144428,0.037843,0.144428,21.260,26.530,15.99,5.821667,0.738000,8.631538,0.720667,17.207647,3.983750,18.918421,6.878889,36.038621,117.591571,506.229980,5306.025387,738.794180,6,207.641691,201.821006,178.733735,0.0,1.815834,31,84.0,30,84.0,30,84.0,10,10,1,6,6,2,0,114,0,0
3,11.96539,45.17531,2023-06-16,veneto,agna,16,6,24,2023,-0.101168,-0.994869,1.224647e-16,-1.0,0.292057,-0.956401,"3,400",46.73306,0.332663,-0.000942,-0.340409,0.000942,0.423918,0.096957,-0.401135,-0.096957,0.083458,0.070568,0.060930,0.070568,25.490,32.430,18.55,4.870000,0.610000,7.752222,-2.034615,15.791176,4.124000,21.578571,5.119231,25.195119,107.229231,1181.997877,16631.850238,1165.197945,3,121.193871,0.122498,180.188821,0.0,3.769479,31,99.0,36,99.0,30,99.0,12,12,1,6,7,2,0,193,0,0
4,12.04755,46.30297,2023-06-16,veneto,agordo,16,6,24,2023,-0.101168,-0.994869,1.224647e-16,-1.0,0.292057,-0.956401,"4,249",47.84463,0.786919,0.326298,-0.654265,-0.326298,0.747121,0.333629,-0.614642,-0.333629,0.025465,0.022967,0.019821,0.022967,14.110,16.070,12.15,1.921429,-4.077692,5.623333,-2.298000,9.285385,-0.998333,11.544545,0.444545,52.359633,83.830531,911.606543,15023.566702,684.464446,18,124.001040,1470.386475,152.701532,0.0,2.289867,15,91.0,10,97.0,10,97.0,5,5,7,1,1,2,0,22,0,0


In [7]:
X_test = data_test.select_dtypes(exclude=['object']).drop(columns = ['case'])
y_test = data_test['case']

X_test = scaler.transform(X_test)
X_test = imputer.fit_transform(X_test)

results_test = inference_lin_model(model, data_test, X_test, y_test)


In [8]:
results_test.drop(columns='case', inplace= True)
results_test

,x,y,lau1,day,month,year,score
0,11.73897,44.96915,polesella,16,6,2023,1.754431e-01
1,11.87982,45.22842,conselve,16,6,2023,7.690437e-02
2,11.68127,45.11042,barbona,16,6,2023,6.222191e-02
3,11.96375,45.41220,noventa padovana,16,6,2023,5.843208e-02
4,11.79794,45.06527,rovigo,16,6,2023,4.676810e-02
...,...,...,...,...,...,...,...
566,11.93045,46.43842,rocca pietore,16,6,2023,9.275098e-09
567,12.42616,46.50321,lozzo da cadore,16,6,2023,8.378370e-09
568,11.85565,46.37360,falcade,16,6,2023,4.461875e-09
569,11.39118,45.89987,rotzo,16,6,2023,3.374582e-09


In [9]:
bins_path = f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_Bins_{YEAR}.csv'

bins = []

with open(bins_path, mode='r', newline='') as file:
    reader = csv.reader(file)
    for row in reader:
        bins.extend(map(float, row))

print(bins)

FileNotFoundError: [Errno 2] No such file or directory: '../../data/Veneto/IT_Veneto_Bins_2023.csv'

In [ ]:
results_test['risk_class'] = pd.cut(results_test['score'], bins=bins, labels=False, include_lowest=True)
results_test

In [ ]:
results_test.to_csv(f"../../data/{NUTS2}/results/{NUTS0}_{NUTS2}_Results_{YEAR}-{MONTH}-{PERIOD}_(new).csv", encoding = enc, index = False)

In [ ]:
##TODO Visualisation of results

,municipality,geometry,x,y,day,month,year,probability
0,chioggia,"POLYGON ((12.29589 45.33225, 12.29961 45.30769...",12.24756,45.27153,1,5,2023,0.003282
1,venezia,"POLYGON ((12.58835 45.53969, 12.58795 45.53951...",12.32478,45.43497,1,5,2023,0.002148
2,codevigo,"POLYGON ((12.12752 45.30091, 12.12959 45.30046...",12.18085,45.26512,1,5,2023,0.001883
3,rosolina,"POLYGON ((12.32883 45.14614, 12.32880 45.14586...",12.30160,45.08601,1,5,2023,0.000718
4,cavallino treporti,"POLYGON ((12.51153 45.50279, 12.51263 45.50274...",12.49633,45.47024,1,5,2023,0.000645
